In [10]:
# reprompt normal
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("to_reprompt_stt.tsv", sep="\t", dtype=str)

# df = pd.concat(dfs, ignore_index=True)  # appends; duplicates are kept
df["max_score"] = df["max_score"].str.replace(",", ".").astype(float)
df["human_score"] = df["human_score"].str.replace(",", ".").astype(float)
df["error_type"] = df["error_type"].astype(int)
df["rueckfrage_human"] = df["rueckfrage_human"].astype(int)

df = df.rename(columns={
    "max_score": "max_points",
    "rueckfrage_human": "rueckfrage",
    "correct_answer": "answer_de"})

# check for column names: question_text, student_answer, gold_answer, maximal_score, human_score, human_feedback
df.head()

,question_de,student_answer,answer_de,keywords,max_points,human_score,human_feedback,rueckfrage,model,transkript_stt_model,error_type
0,Was ist der Unterschied zwischen Klassifikatio...,Bei der Klassifikation handelt es sich um eine...,Klassifikation ist eine überwachte Lernaufgabe...,Klassifikation; überwachte Lernens; Klassen vo...,5.0,5.0,Die Antwort stellt sowohl Klassifikation als a...,0,mistral-small3.1:latest,bei der Klassifikation handelt sich um eine Au...,1
1,Was ist der Unterschied zwischen Q-Learning un...,Q-Learning ist ein modellfreier Reinforcement-...,Q-Learning ist ein modellfreies Reinforcement-...,Q-Learning; modellfrei; Q-Wertefunktion; Polic...,5.0,4.0,Die Antwort erklärt Q-Learning und Policy-Grad...,0,mistral-small3.1:latest,Q Learning ist ein Modell freier Reinforcement...,0
2,Was ist der Zweck von Heatmaps in der EDA?,"Heatmaps werden in der EDA verwendet, um die M...",Heatmaps in der EDA visualisieren die Magnitud...,Heatmaps; EDA; Magnitude; Stärke einer Beziehu...,4.0,0.0,Die Antwort ist fachlich falsch: Heatmaps dien...,3,mistral-small3.1:latest,"Heatmaps werden in der Idea verwendet, um die ...",2
3,Was ist Regularisierung in neuronalen Netzwerken?,Regularisierung in neuronalen Netzwerken beinh...,Regularisierung in neuronalen Netzwerken beinh...,Regularisierung; neuronale Netzwerke; Verlustf...,6.0,6.0,1) Die Antwort wurde fachlich korrekt dargeste...,0,deepseek-r1:8b,Regularisierung in neuronalen Netzwerken beinh...,3
4,Was ist der Zweck von A/B-Tests im Data Science?,A/B-Tests ermöglichen einen Vergleich zwischen...,"A/B-Testing wird verwendet, um zwei oder mehr ...",A/B-Testing; Produkte oder Interventionen; vor...,6.0,6.0,Die Antwort stellt den Zweck von A/B-Tests fac...,0,phi4:latest,abetests ermöglichen ein Vergleich zwischen Pr...,1


In [11]:
from llm_handler import LLMHandler
import json

# TODO: Write prompt for evaluation
prompt = """
Du bekommst eine Prüfungsfrage, die Antwort eines Studenten darauf, die Musterlösung zu der Frage und die maximal erreichbare Punktzahl für eine richtige Antwort.
Deine Aufgabe ist es, die Antwort des Studenten mit der Musterlösung zu vergleichen und diese zu bewerten.

Dabei sollst du dich an folgendem Leitfaden orientieren:
1) Hat der Student den Sachverhalt fachlich korrekt dargestellt, ohne wesentliche Fehler oder falsche Zusammenhänge?
2) Verwendet der Student die relevanten Schlüsselbegriffe korrekt und im richtigen Kontext? Schau dabei auf folgende Liste der wichtigsten Fachbegriffe: {keywords}.
3) Geht der Student auf alle wesentlichen Aspekte der Fragestellung ein oder bleiben zentrale Punkte unbeantwortet?
4) Werden die angesprochenen Konzepte klar voneinander unterschieden und nicht miteinander vermischt?
5) Ist die Antwort logisch aufgebaut, nachvollziehbar formuliert und für den Prüfer gut verständlich?
6) Soll vom Prüfer eine Rückfrage gestellt werden, um auf Lücken zu prüfen?

Deine Ausgabe soll aus folgenden Elementen bestehen:
- **llm_feedback**: In diesem Feld schreibst du die Bewertung zur Antwort des Studenten. Dabei gehst du auf jeden der oben genannten Punkte ein und formulierst anschließend einen Fließtext daraus. Beschränke dich bei dem Fließtext auf maximal 4-5 Sätze.
- **llm_rating**: Hier vergibst du die Punktzahl, die deiner Meinung nach die Antwort des Studenten wiederspiegelt. Du kannst die Punkte in 0,5 Schritten vergeben. Die maximal vergebbare Punktzahl ist {max_score}.
- **rueckfrage**: Du füllst dieses Feld mit einem Integer aus. Dieser Integer kann 0, 1, 2 oder 3 sein. 0 bedeutet, dass keine Rückfrage notwendig ist. 1 bedeutet, dass eine Rückfrage notwendig ist, die sich auf nicht genannte Fach- bzw. Schlüsselbegriffe bezieht. 2 bedeutet, dass eine Rückfrage gestellt wird und sich auf eine fehlende Teilantwort bezieht. 3 bedeutet, dass sich die gestellte Rückfrage auf einem Teil der falschen Antwort bezieht.

Frage:
{question}

Studentenantwort:
{student_answer}

Musterlösung:
{correct_answer}

Für diese Frage gibt es maximal {max_score} Punkte.

<Antwortformat>
```json
{{
"llm_feedback": "<Hier antwortest du auf die Antwort des Studenten und gibts ihm Feedback entsprechend der Analysepunkte. Die Antwort ist ein Text, es gibt keine JSON-Struktur>",
"llm_rating": "<Gesamtrating, maximale Punkte: {max_score}, gerundet auf eine Nachkommastelle>",
"rueckfrage": "<0 | 1 | 2 | 3>"
}}```
"""

In [12]:
import time

def _results_current_model(prompt: str, df: pd.DataFrame) -> pd.DataFrame:
  llm_results = []
  for _, row in df.iterrows():
    llm = LLMHandler(model=row["model"])
    prompt_filled = prompt.format(
      question=row['question_de'],
      student_answer=row['transkript_stt_model'],
      keywords=row["keywords"],
      correct_answer=row['answer_de'],
      max_score=row['max_points']
    )
    while True:
      try:
        print(f"Try processing row {_}")
        answer = llm.call_llm(prompt=prompt_filled)
        if not isinstance(answer, dict):
          raise ValueError("LLM response is not a dictionary")
        answer["question"] = row['question_de']
        answer["student_answer"] = row['transkript_stt_model']
        answer["correct_answer"] = row['answer_de']
        answer["keywords"] = row["keywords"]
        answer["max_score"] = row['max_points']
        answer["human_score"] = row['human_score']
        answer["human_feedback"] = row['human_feedback']
        answer["rueckfrage_human"] = row["rueckfrage"]
        answer["model"] = row["model"]
        llm_results.append(answer)     
        with open("checkpoint_transcript.json", "a", encoding="utf-8") as f:
          f.write(json.dumps(answer, ensure_ascii=False) + "\n") 
        break
      except Exception as e:
        print(f"Error processing row {_}: {e}. Retrying in 5 seconds...")
        time.sleep(5)
  return pd.DataFrame(llm_results)

In [ ]:
final_df = _results_current_model(
    prompt=prompt,
    df=df
)
final_df.to_csv("reprompt_results_stt_third_try.csv", index=False)

Try processing row 0
Try processing row 1
Try processing row 2
Try processing row 3
Try processing row 4
Try processing row 5
Try processing row 6
Try processing row 7
Try processing row 8
